In [1]:
import pandas as pd
import numpy as np
import requests
import socket
import re
import os
import time
import tldextract
import whois
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from difflib import SequenceMatcher
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

print("All imports done")

All imports done


In [2]:
df = pd.read_csv("../data/raw/urls_with_labels.csv")
print(f"Loaded {len(df)} URLs")
print(df["label"].value_counts())

Loaded 30000 URLs
label
1    15000
0    15000
Name: count, dtype: int64


In [3]:
KNOWN_BRADS = [
    'google', 'microsoft', 'apple', 'amazon', 'paypal',
    'facebook', 'twitter', 'instagram', 'netflix', 'ebay',
    'linkedin', 'dropbox', 'github', 'yahoo', 'gmail'
]

SHORTENING_SERVICES = [
    'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly',
    'buff.ly', 'adf.ly', 'bit.do', 'cutt.ly'
]

SUSPICIOUS_TLDS = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.work']

PHISHING_KEYWORDS = ['login', 'verify', 'secure', 'account', 'update',
                    'banking', 'confirm', 'password', 'signin', 'webscr',
                    'ebayisapi', 'payment']

def is_ip_address(hostname):
    try:
        socket.inet_aton(hostname)
        return 1
    except:
        return 0

def brand_similarity_score(domain):
    domain_clean = domain.replace('www.', '').split('.')[0].lower()
    max_sim = 0
    for brand in KNOWN_BRADS:
        if domain_clean == brand:
            return 0.0
        sim = SequenceMatcher(None, domain_clean, brand).ratio()
        max_sim = max(max_sim, sim)
    return round(max_sim, 3)

def extract_lexical_features(url):
    try:
        parsed = urlparse(url)
        ext = tldextract.extract(url)
        hostname = parsed.hostname or ""
        path = parsed.path or ""
        full_url = url.lower()
        features = {
            "url_length": len(url),
            "hostname_length": len(hostname),
            "nb_dots": url.count('.'),
            "nb_hyphens": url.count('-'),
            "nb_slash": url.count('/'),
            "nb_at": url.count('@'),
            "nb_qmark": url.count('?'),
            "nb_equal": url.count('='),
            "nb_underscore": url.count('_'),
            "nb_percent": url.count('%'),
            "nb_digits": sum(c.isdigit() for c in url),
            "ratio_digits_url": sum(c.isdigit() for c in url) / max(len(url), 1),
            "ratio_digits_host": sum(c.isdigit() for c in hostname) / max(len(hostname), 1),
            "https_token": 1 if parsed.scheme == 'https' else 0,
            "ip_in_hostname": is_ip_address(hostname),
            "nb_subdomains": len(ext.subdomain.split('.')) if ext.subdomain else 0,
            "prefix_suffix": 1 if '-' in ext.domain else 0,
            "suspicious_tld": 1 if any(url.endswith(tld) or f"{tld}/" in url for tld in SUSPICIOUS_TLDS) else 0,
            "shortening_service": 1 if any(s in hostname for s in SHORTENING_SERVICES) else 0,
            "phishing_hints": 1 if any(kw in full_url for kw in PHISHING_KEYWORDS) else 0,
            "brand_similarity": brand_similarity_score(ext.domain),
            "path_length": len(path),
            "has_port": 1 if parsed.port else 0,
            "double_slash_in_path": 1 if '//' in path else 0,
        }
        return features
    except Exception as e:
        return {}


In [4]:
def extract_whois_features(url):
    default = {'domain_age_days': -1, 'domain_registration_length': -1, 'dns_record': 0}
    try:
        ext = tldextract.extract(url)
        domain = f'{ext.domain}.{ext.suffix}'

        w = whois.whois(domain)

        dns_record = 1

        creation = w.creation_data
        if isinstance(creation, list):
            creation = creation[0]
        age = (datetime.now() - creation).days if creation else -1

        expiry = w.expiration_date
        if isinstance(expiry, list):
            expiry = expiry[0]
        reg_length = (expiry - creation).days if(expiry and creation) else -1

        return {
            "domain_age_days": age,
            "domain_registration_length": reg_length,
            "dns_record": dns_record
        }
    except:
        return default

In [5]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def extract_webpage_features(url):
    default = {
        "nb_hyperlinks": -1, "ratio_internal_links": -1,
        "nb_external_css": -1, "has_login_form": 0,
        "has_iframe": 0, "external_favicon": 0,
        "title_matches_domain": 0
    }
    try:
        response = requests.get(url, timeout=5, headers=HEADERS, allow_redirects=True)
        soup = BeautifulSoup(response.text, "html.parser")
        parsed = urlparse(url)
        ext = tldextract.extract(url)
        base_domain = f"{ext.domain}.{ext.suffix}"

        all_links = soup.find_all("a", href=True)
        nb_links = len(all_links)
        internal = sum(1 for a in all_links if base_domain in a["href"] or a["href"].startswith('/'))
        ratio_internal = internal / max(nb_links, 1)

        css_links = soup.find_all("link", rel=lambda r: r and "stylesheet" in r)
        external_css = sum(1 for l in css_links if l.get("href") and base_domain not in l.get("href", ""))

        inputs = soup.find_all("input")
        has_login = 1 if any(i.get("type") == "password" for i in inputs) else 0

        has_iframe = 1 if soup.find("iframe") else 0

        favicon = soup.find("link", rel=lambda r: r and "icon" in r)
        ext_favicon = 0
        if favicon and favicon.get("href"):
            ext_favicon = 1 if (favicon["href"].startswith("http") and base_domain not in favicon["href"]) else 0

        title = soup.title.string if soup.title else ""
        title_match = 1 if (title and ext.domain.lower() in title.lower()) else 0

        return {
            "nb_hyperlinks": nb_links,
            "ratio_internal_links": round(ratio_internal, 3),
            "nb_external_css": external_css,
            "has_login_form": has_login,
            "has_iframe": has_iframe,
            "external_favicon": ext_favicon,
            "title_matches_domain": title_match
        }
    except:
        return default

In [6]:
def extract_all_features(url, label):
    features ={"url": url, "label": label}
    features.update(extract_lexical_features(url))
    features.update(extract_whois_features(url))
    features.update(extract_webpage_features(url))
    return features

In [7]:
BATCH_SIZE = 500
SAVE_DIR = '../data/batches'

completed = set()

for f in os.listdir(SAVE_DIR):
    if f.startswith('batch_') and f.endswith('.csv'):
        idx = int(f.replace('batch_', "").replace('.csv', ''))
        completed.add(idx)

print(f'Already completed batches: {sorted(completed)}')
print(f'Total URLs: {len(df)}')
print(f'Starting extraction... This will take several hours')

urls = df['url'].tolist()
labels = df['label'].tolist()

for batch_start in range(0, len(df), BATCH_SIZE):
    if batch_start in completed:
        print(f'Skipping batch {batch_start} (already done)')
        continue

    batch_urls = urls[batch_start:batch_start + BATCH_SIZE]
    batch_labels = labels[batch_start:batch_start + BATCH_SIZE]

    batch_features = []

    for i, (url, label) in enumerate(zip(batch_urls, batch_labels)):
        row = extract_all_features(url, label)
        batch_features.append(row)

        if(i + 1) % 100 == 0:
            print(f'Batch {batch_start} : {i + 1}/ {len(batch_urls)} done')

    batch_df = pd.DataFrame(batch_features)
    batch_df.to_csv(f'{SAVE_DIR}/batch_{batch_start}.csv', index=False)
    print(f'Saved batch_{batch_start}.csv ({len(batch_df)} rows)')

    time.sleep(1)

print('All batches complete!')


Already completed batches: [0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500, 7000, 7500, 8000, 8500, 9000, 9500, 10000, 10500, 11000, 11500, 12000, 12500, 13000, 13500, 14000, 14500, 15000, 15500, 16000, 16500, 17000, 17500, 18000, 18500, 19000, 19500, 20000, 20500, 21000, 21500, 22000, 22500, 23000, 23500, 24000, 24500, 25000, 25500, 26000, 26500, 27000, 27500, 28000, 28500, 29000, 29500]
Total URLs: 30000
Starting extraction... This will take several hours
Skipping batch 0 (already done)
Skipping batch 500 (already done)
Skipping batch 1000 (already done)
Skipping batch 1500 (already done)
Skipping batch 2000 (already done)
Skipping batch 2500 (already done)
Skipping batch 3000 (already done)
Skipping batch 3500 (already done)
Skipping batch 4000 (already done)
Skipping batch 4500 (already done)
Skipping batch 5000 (already done)
Skipping batch 5500 (already done)
Skipping batch 6000 (already done)
Skipping batch 6500 (already done)
Skipping batch 7000 

In [9]:
import glob

batch_files = sorted(glob.glob("../data/batches/batch_*.csv"))
print(f"Found {len(batch_files)} batch files")

all_batches = [pd.read_csv(f) for f in batch_files]
final_df = pd.concat(all_batches, ignore_index=True)

print(f"Total rows: {len(final_df)}")
print(f"Columns: {len(final_df.columns)}")
print(f"Class balance:\n{final_df['label'].value_counts()}")

final_df.to_csv("../data/processed/final_dataset.csv", index=False)
print("Saved final_dataset.csv")
final_df.head()

Found 60 batch files
Total rows: 30000
Columns: 36
Class balance:
label
1    15000
0    15000
Name: count, dtype: int64
Saved final_dataset.csv


,url,label,url_length,hostname_length,nb_dots,nb_hyphens,nb_slash,nb_at,nb_qmark,nb_equal,...,domain_age_days,domain_registration_length,dns_record,nb_hyperlinks,ratio_internal_links,nb_external_css,has_login_form,has_iframe,external_favicon,title_matches_domain
0,http://yaho-world-wandering-limit-8b7e.yorzeko...,1,61,54,3,4,2,0,0,0,...,-1,-1,0,2,0.000,1,0,0,0,0
1,https://westlaw.com,0,19,11,1,0,2,0,0,0,...,-1,-1,0,1,1.000,0,0,0,0,0
2,https://google.mg,0,17,9,1,0,2,0,0,0,...,-1,-1,0,20,0.250,1,0,0,0,1
3,https://charlotte.edu,0,21,13,1,0,2,0,0,0,...,-1,-1,0,94,0.862,2,0,0,0,1
4,https://mesesemondre.web.app,1,28,20,2,0,2,0,0,0,...,-1,-1,0,2,0.000,1,0,0,0,0


In [10]:
df = pd.read_csv('../data/processed/final_dataset.csv')
print(f'Total rows: {len(df)}')
print(f'Columns: {len(df.columns)}')
print(f'Class balance: {df['label'].value_counts()}')
print(f'Missing values: {df.isnull().sum().sort_values(ascending=False).head(10)}')

Total rows: 30000
Columns: 36
Class balance: label
1    15000
0    15000
Name: count, dtype: int64
Missing values: url                0
label              0
url_length         0
hostname_length    0
nb_dots            0
nb_hyphens         0
nb_slash           0
nb_at              0
nb_qmark           0
nb_equal           0
dtype: int64
